# Phase 1: Data Audit & Contracts

In [ ]:
import sys
sys.path.append("..")

from src.config import *

# Load & register data
df_raw = load_raw_data(url="YOUR_URL_HERE")
register_duckdb_table(df_raw, table_name="trips")

print("Config loaded successfully.")

## Step 1.1: Schema Inspection
Inspect the dataset's structure: column names, data types and a statistical summary.
This gives us a complete inventory of what we are working with before any auditing or cleaning begins.

In [ ]:
# ------------------------------------------------------------
# Step 1.1: Schema Inspection
# ------------------------------------------------------------

def inspect_schema(df: pd.DataFrame) -> pd.DataFrame:
    """
    Produce a structured schema inventory for the dataset.

    For each column, reports the data type, number of non-null values,
    null count, null percentage and number of unique values.

    Parameters
    ----------
    df: pd.DataFrame
        The DataFrame to inspect.

    Returns
    -------
    pd.DataFrame
        Schema summary table sorted by null percentage descending.
    """
    schema = pd.DataFrame({
        "dtype": df.dtypes,
        "non_null": df.notna().sum(),
        "null_count": df.isna().sum(),
        "null_pct": (df.isna().sum() / len(df) * 100).round(2),
        "unique_values": df.nunique()
    })
    return schema.sort_values("null_pct", ascending=False)

# Schema table
schema_df = inspect_schema(df_raw)

print("\n" + "=" * 70)
print("Dataset Shape")
print("=" * 70)
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")

print("\n" + "=" * 70)
print("Schema Inventory")
print("=" * 70)
print(schema_df.to_string())

print("\n" + "=" * 70)
print("Statistical Summary (numeric columns)")
print("=" * 70)
df_raw.describe().T


## Step 1.2: Missing Values Audit
Quantify and visualize missing values across all columns. Understanding the extend and pattern of missingness
determines whether we drop, impute or flag affected rows in Phase 3 (Data Cleaning).

In [ ]:
# ------------------------------------------------------------
# Step 1.2: Missing Values Audit
# ------------------------------------------------------------

def audit_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute a missing values summary for every column in the DataFrame.

    Only columns with at least one missing value are included in the 
    returned summary table.

    Parameters
    ----------
    df: pd.DataFrame
        The DataFrame to audit.

    Returns
    -------
    pd.DataFrame
        Summary table with columns: null_count, null_pct, dtype.
        Returns an empty DataFrame if no missing values are found.
    """
    missing = pd.DataFrame({
        "null_count": df.isna().sum(),
        "null_pct": (df.isna().sum() / len(df) * 100).round(4),
        "dtype": df.dtypes
    })

    missing = missing[missing["null_count"] > 0].sort_values(
        "null_pct", ascending=False
    )

    return missing

def plot_missing_values(missing: pd.DataFrame) -> None:
    """
    Plot a horizontal bar chart of missing value percentages per column.

    If no columns have missing values, logs a confirmation message and
    skips plotting.

    Parameters
    ----------
    missing: pd.DataFrame
        Output of ``audit_missing_values()``.
    """
    if missing.empty:
        logger.info("No missing values found. Skipping plot.")
        return

    fig, ax = plt.subplots(figsize=(10, max(4, len(missing) * 0.5)))
    ax.barh(
        missing.index,
        missing["null_pct"],
        color=PALETTE["accent"]
    )
    ax.set_xlabel("Missing (%)")
    ax.set_title("Missing Values by Column")
    ax.invert_yaxis()

    # Annotate bars with exact percentages
    for i, (idx, row) in enumerate(missing.iterrows()):
        ax.text(
            row["null_pct"] + 0.01,
            i,
            f"{row['null_pct']:.2f}% ({int(row['null_count']):,})",
            va="center",
            fontsize=9
        )
        
    plt.tight_layout()
    save_figure(fig, "phase1_missing_values")
    plt.show()

# Run audit
missing_df = audit_missing_values(df_raw)

print("=" * 70)
print("Missing Values Summary")
print("=" * 70)

if missing_df.empty:
    print("  ✅ No missing values detected across all columns.")
else:
    print(f"  ⚠️ {len(missing_df)} column(s) contain missing values.\n")
    print(missing_df.to_string())

plot_missing_values(missing_df)


## Step 1.2: Missing Values Audit ✅

No missing values detected across all 17 columns — no imputation required.

However, the schema inspection revealed several issues worth flagging for cleaning:

| Column | Issue |
|--------|-------|
| `extra` | dtype is `object` — should be `float64` |
| `total_amount` | dtype is `object` — should be `float64` |
| `fare_amount` | min = **-233.0** — negative fares are invalid |
| `tip_amount` | min = **-4.52** — negative tips are invalid |
| `tolls_amount` | min = **-5.76** — negative tolls are invalid |
| `mta_tax` | min = **-0.50** — negative tax is invalid |
| `ratecodeid` | max = **99** — valid range is 1–6 per data dictionary |
| `passenger_count` | min = **0** — trips with zero passengers are suspicious |
| `trip_distance` | min = **0.0** — zero distance trips need investigation |

These will all be addressed in Phase 3 — Data Cleaning.

## Step 1.3: Duplicate Detection
Check for exact duplicate rows and near-duplicate trips, same pickup time, dropoff time, and fare amount,
which may indicate data entry errors or system recording issues.

In [ ]:
# ------------------------------------------------------------
# Step 1.3: Duplicate Detection
# ------------------------------------------------------------

def audit_duplicates(df: pd.DataFrame) -> dict:
    """
    Detect exact and near-duplicate records in the dataset.

    Exact duplicates: rows identical across all columns.
    Near duplicates: rows sharing the same pickup time, droposs time and fare amount
                        likely the same trip recorded twice

    Parameters
    ----------
    df: pd.DataFrame
        The DataFrame to audit

    Returns
    -------
    dict
        Summary containing exact_count, near_count and the near-duplicates DataFrame for further inspection.
    """
    # Exact duplicates: identical across every column
    exact_mask = df.duplicated(keep=False)
    exact_count = exact_mask.sum()

    # Near duplicates: same pickup, dropoff and fare
    near_keys = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "fare_amount"]
    near_mask = df.duplicated(subset=near_keys, keep=False)
    near_count = near_mask.sum()

    logger.info(f"Exact duplicates: {exact_count:,}")
    logger.info(f"Near duplicates: {near_count:,} "
               f"({near_count / len(df) * 100:.2f}% of dataset)")

    return {
        "exact_count": exact_count,
        "near_count": near_count,
        "near_df": df[near_mask].sort_values(near_keys)
    }

def plot_duplicate_profile(near_df: pd.DataFrame) -> None:
    """
    Visualize near-duplicate trips by vendor to identify systematic patterns.

    Parameters
    ----------
    near_df: pd.DataFrame
        Near-duplicate subset returned by ``audit_duplicates()``.
    """
    if near_df.empty:
        logger.info("No near-duplicates to plot.")
        return

    vendor_map = {1: "Creative Mobile Tech", 2: "VeriFone Inc."}
    counts = (
        near_df["vendorid"]
        .map(vendor_map)
        .value_counts()
        .reset_index()
    )
    counts.columns = ["vendor", "count"]

    fig, ax = plt.subplots()
    ax.bar(counts["vendor"], counts["count"], color=PALETTE["accent"], width=0.4)
    ax.set_title("Near-Duplicate Trips by Vendor")
    ax.set_ylabel("Number of Duplciate Trips")
    ax.set_xlabel("Vendor")
    for i, row in counts.iterrows():
        ax.text(i, row["count"] + 10, f"{row['count']:,}", ha="center", fontsize=10)

    plt.tight_layout()
    save_figure(fig, "phase1_duplicates_by_vendor")
    plt.show()

# Run audit
dup_summary = audit_duplicates(df_raw)

print("\n" + "=" * 70)
print("Duplicate Summary")
print("=" * 70)

print(f"  Exact duplicates: {dup_summary['exact_count']:,}")
print(f"  Near duplicates: {dup_summary['near_count']:,}")

print("\n" + "=" * 70)
print("Near-Duplicate Sample (top 6)")
print("=" * 70)

print(dup_summary["near_df"][
    ["tpep_pickup_datetime", "tpep_dropoff_datetime",
    "fare_amount", "vendorid", "passenger_count"]
].head(6).to_string(index=False))

plot_duplicate_profile(dup_summary["near_df"])


## Step 1.3 — Duplicate Detection ✅

No exact duplicates found. However, **518 near-duplicate trips** were detected
(same pickup time, dropoff time, and fare amount).

| Metric | Value |
|---|---|
| Exact duplicates | 0 |
| Near duplicates | 518 (0.05% of dataset) |

**Near-duplicates by vendor:**

| Vendor | Near-Duplicate Trips |
|---|---|
| VeriFone Inc. | 302 |
| Creative Mobile Tech | 216 |

**Observations:**
- Near-duplicates appear across **both vendors**, suggesting this is not a
  vendor-specific recording issue
- Some near-duplicates share the same `vendorid` (same vendor recorded the trip
  twice), while others differ in `vendorid` (same trip recorded by both vendors)
- At 0.05% of the dataset, the volume is small but will be flagged and removed
  in Phase 3 — Data Cleaning

## Step 1.4: Value Range Validation
Validate each column against the business rules defined in the TLC data dictionary.
Flag columns where values fall outside expected ranges.
These flags define the cleaning contracts enforced in Phase 3.

In [ ]:
# ------------------------------------------------------------
# Step 1.4: Value Range Validation
# ------------------------------------------------------------

# Business rules from the TLC data dictionary
RANGE_CONTRACTS: dict = {
    # column : (min_valid, max_valid, description)
    "passenger_count": (1, 9, "At least 1 passenger, max 9"),
    "trip_distance": (0.01, 200, "Must be > 0, cap at 200 miles"),
    "ratecodeid": (1, 6, "Valid codes are 1-6 per data dictionary"),
    "fare_amount": (0.01, 1000, "Must be positive"),
    "extra": (0, 10, "Valid surcharges: $0, $0.50, $1"),
    "mta_tax": (0, 0.5, "Fixed at $0.50 or $0"),
    "tip_amount": (0, 500, "Cannot be negative"),
    "tolls_amount": (0, 500, "Cannot be negative"),
    "improvement_surchange": (0, 0.3, "Fixed at $0.30 or $0"),
    "total_amount": (0.01, 2000, "Must be positive")
}

def validate_ranges(df: pd.DataFrame, contracts: dict) -> pd.DataFrame:
    """
    Validate numeric columns against defined business rule contracts.

    For each contract, counts the number of rows that violate the 
    minimum or maximum bound and computes the violation rate.

    Parameters
    ----------
    df: pd.DataFrame
        The DataFrame to validate.
    contracts: dict
        Dictionary mapping column names to (min_valid, max_valid, description)
        tuples defining the acceptable value range.

    Returns
    -------
    pd.DataFrame
        Validation report with one row per contract, showing violation
        counts and percentages. Sorted by violation count descending.
    """
    records = []
    for col, (low, high, desc) in contracts.items():
        if col not in df.columns:
            logger.warning(f"Column '{col}' not found. Skipping...")
            continue

        # Coerce to numeric to handle object-typed columns
        series = pd.to_numeric(df[col], errors="coerce")
        n_below = (series < low).sum()
        n_above = (series > high).sum()
        n_violation = n_below + n_above
        pct = round(n_violation / len(df) * 100, 4)

        records.append({
            "column": col,
            "rule": desc,
            "min_valid": low,
            "max_valid": high,
            "below_min": int(n_below),
            "above_max": int(n_above),
            "violations": int(n_violation),
            "violation_pct": pct
        })

    report = pd.DataFrame(records).sort_values("violations", ascending=False)

    return report

def plot_violations(report: pd.DataFrame) -> None:
    """
    Plot a horizontal bar chart of range violation counts per column.

    Only columns with at least one violation are included.

    Parameters
    ----------
    report: pd.DataFrame
        Output of ``validate_ranges()``.
    """
    flagged = report[report["violations"] > 0]
    if flagged.empty:
        logger.info("No range violations found. Skipping plot...")
        return

    fig, ax = plt.subplots(figsize=(10, max(4, len(flagged) * 0.6)))
    bars = ax.barh(
        flagged["column"],
        flagged["violations"],
        color=PALETTE["accent"]
    )
    ax.set_xlabel("Number of Violations")
    ax.set_title("Range Violations by Column")
    ax.invert_yaxis()

    for i, (_, row) in enumerate(flagged.iterrows()):
        ax.text(
            row["violations"] + 50,
            i,
            f"{row['violations']:,} ({row['violation_pct']}%)",
            va="center",
            fontsize=9
        )

    plt.tight_layout()
    save_figure(fig, "phase1_range_violations")
    plt.show()

# Run validation
validation_report = validate_ranges(df_raw, RANGE_CONTRACTS)

print("\n" + "=" * 70)
print("Range Validation Report")
print("=" * 70)

print(validation_report.to_string(index=False))

print(f"\n Columns with violations:"
     f"{(validation_report['violations'] > 0).sum()} /"
     f"{len(validation_report)}")

plot_violations(validation_report)


## Step 1.4 — Value Range Validation ✅

All 9 numeric columns validated against TLC data dictionary business rules.
Every column returned at least one violation.

| Column | Violations | Rate | Issue |
|---|---|---|---|
| `trip_distance` | 7,049 | 0.70% | Zero or near-zero distance trips |
| `fare_amount` | 893 | 0.09% | Negative fares |
| `total_amount` | 724 | 0.07% | Negative total amounts |
| `mta_tax` | 564 | 0.06% | Negative or above $0.50 cap |
| `extra` | 263 | 0.03% | Negative or above valid surcharge range |
| `passenger_count` | 88 | 0.009% | Zero passenger trips |
| `ratecodeid` | 15 | 0.002% | Invalid code 99 (valid range: 1–6) |
| `tip_amount` | 9 | 0.0009% | Negative tips |
| `tolls_amount` | 6 | 0.0006% | Negative tolls |

**Observations:**
- `trip_distance` is the most violated column — 7,049 trips with zero or
  near-zero distance are likely meter errors or cancelled trips
- Negative values in `fare_amount`, `tip_amount`, and `tolls_amount` likely
  represent refunds or system corrections and must be removed
- `ratecodeid = 99` is completely outside the valid range of 1–6 and has no
  business meaning — these 15 rows will be dropped in Phase 3
- Total violations across all columns are well under 1% of the dataset,
  so cleaning will not significantly reduce the training data size

In [ ]:
# ------------------------------------------------------------
# Step 1.5: Data Contract Summary
# ------------------------------------------------------------

def build_data_contract(
    df: pd.DataFrame,
    validation_report: pd.DataFrame,
    dup_summary: dict,
) -> pd.DataFrame:
    """
    Consolidate Phase 1 audit findings into a formal data contract table.

    Each row represents one cleaning rule to be enforced in Phase 3,
    including the issue type, affected column, rule description, estimated
    rows impacted, and the planned action.

    Parameters
    ----------
    df: pd.DataFrame
        The raw DataFrame (used for row count reference).
    validation_report : pd.DataFrame
        Output of ``validate_ranges()`` from Step 1.4.
    dup_summary: dict
        Output of ``audit_duplicates()`` from Step 1.3.

    Returns
    -------
    pd.DataFrame
        Formal data contract with one rule per row.
    """
    n_total = len(df)

    contracts = [
        # Dtype fixes
        {
            "issue": "Wrong dtype",
            "column": "extra",
            "rule": "Cast to float64",
            "rows_impacted": n_total,
            "action": "CAST",
        },
        {
            "issue": "Wrong dtype",
            "column": "total_amount",
            "rule": "Cast to float64",
            "rows_impacted": n_total,
            "action": "CAST",
        },
        # Duplicates
        {
            "issue": "Near duplicates",
            "column": "all",
            "rule": "Same pickup, dropoff, fare — keep first occurrence",
            "rows_impacted": dup_summary["near_count"],
            "action": "DROP",
        },
        # Range violations
        {
            "issue": "Zero distance trips",
            "column": "trip_distance",
            "rule": "trip_distance must be >= 0.01 miles",
            "rows_impacted": 7049,
            "action": "DROP",
        },
        {
            "issue": "Negative fare",
            "column": "fare_amount",
            "rule": "fare_amount must be > 0",
            "rows_impacted": 893,
            "action": "DROP",
        },
        {
            "issue": "Negative total",
            "column": "total_amount",
            "rule": "total_amount must be > 0",
            "rows_impacted": 724,
            "action": "DROP",
        },
        {
            "issue": "Invalid mta_tax",
            "column": "mta_tax",
            "rule": "mta_tax must be between 0 and 0.50",
            "rows_impacted": 564,
            "action": "DROP",
        },
        {
            "issue": "Invalid extra surcharge",
            "column": "extra",
            "rule": "extra must be between 0 and 10",
            "rows_impacted": 263,
            "action": "DROP",
        },
        {
            "issue": "Zero passenger trips",
            "column": "passenger_count",
            "rule": "passenger_count must be >= 1",
            "rows_impacted": 88,
            "action": "DROP",
        },
        {
            "issue": "Invalid rate code",
            "column": "ratecodeid",
            "rule": "ratecodeid must be between 1 and 6",
            "rows_impacted": 15,
            "action": "DROP",
        },
        {
            "issue": "Negative tip",
            "column": "tip_amount",
            "rule": "tip_amount must be >= 0",
            "rows_impacted": 9,
            "action": "DROP",
        },
        {
            "issue": "Negative tolls",
            "column": "tolls_amount",
            "rule": "tolls_amount must be >= 0",
            "rows_impacted": 6,
            "action": "DROP",
        },
    ]

    contract_df = pd.DataFrame(contracts)
    contract_df["pct_of_dataset"] = (
        contract_df["rows_impacted"] / n_total * 100
    ).round(4)

    return contract_df


# Build & display contract
contract_df = build_data_contract(df_raw, validation_report, dup_summary)

print("\n" + "=" * 70)
print("Data Contract")
print("=" * 70)
print(contract_df.to_string(index=False))

# Estimate post-cleaning dataset size
# Note: rows may violate multiple rules so we use SQL to count
# distinct invalid rows rather than summing violations naively
invalid_rows = quick_sql("""
    SELECT COUNT(*) AS invalid_rows FROM trips
    WHERE trip_distance < 0.01
       OR fare_amount <= 0
       OR passenger_count < 1
       OR ratecodeid NOT IN (1,2,3,4,5,6)
       OR tip_amount < 0
       OR tolls_amount < 0
       OR mta_tax < 0
       OR mta_tax > 0.5
""").iloc[0, 0]

est_clean = len(df_raw) - invalid_rows - dup_summary["near_count"]

print(f"\nEstimated Post-Cleaning Dataset Size")
print(f"  Original rows: {len(df_raw):,}")
print(f"  Invalid rows: {invalid_rows:,}")
print(f"  Near duplicates: {dup_summary['near_count']:,}")
print(f"  Estimated clean: {est_clean:,}  "
      f"({est_clean / len(df_raw) * 100:.2f}% retained)")


## Step 1.5 — Data Contract Summary ✅

All Phase 1 audit findings consolidated into a formal data contract.
Phase 3 will enforce these rules in order: cast dtypes → drop duplicates → drop violations.

| Issue | Column | Action | Rows Impacted |
|---|---|---|---|
| Wrong dtype | `extra`, `total_amount` | CAST to float64 | All rows |
| Near duplicates | all | DROP (keep first) | 518 |
| Zero distance trips | `trip_distance` | DROP | 7,049 |
| Negative fare | `fare_amount` | DROP | 893 |
| Negative total | `total_amount` | DROP | 724 |
| Invalid mta_tax | `mta_tax` | DROP | 564 |
| Invalid extra surcharge | `extra` | DROP | 263 |
| Zero passenger trips | `passenger_count` | DROP | 88 |
| Invalid rate code | `ratecodeid` | DROP | 15 |
| Negative tip | `tip_amount` | DROP | 9 |
| Negative tolls | `tolls_amount` | DROP | 6 |

**Post-cleaning size estimate:**

| Metric | Value |
|---|---|
| Original rows | 1,000,000 |
| Invalid rows | 7,742 |
| Near duplicates | 518 |
| **Estimated clean rows** | **991,740 (99.17% retained)** |

> Cleaning removes less than 1% of the dataset — the data is in good shape.
> All violation types are consistent with known TLC data quality issues
> (meter errors, refund corrections, system recording glitches).